# Notebook 02 (Focal Loss Ablation) — XLM-R with Focal Loss

**Project:** Benchmarking Modern Multilingual Transformer Models for Malay-English Code-Switched Sentiment Analysis  
**Authors:** Joshua Joenathan Thomas (25141571) | Amit Kumar Gupta (25109952)

## What this notebook does

This notebook trains XLM-R (`xlm-roberta-base`) twice — with Focal Loss at γ=1.0 and γ=2.0 — and
compares both against the XLM-R v2 cross-entropy baseline.

## Why Focal Loss?

The original transformer experiments used mild linear class weights (factors ~1.01–1.09 for the
three classes). In a deep network these near-unity weights are effectively equivalent to no weighting
at all: the loss surface and gradient magnitudes are almost unchanged. The majority-class (Neutral,
~69 % of test set) continues to dominate training.

Focal Loss (Lin et al. 2017, "Focal Loss for Dense Object Detection") provides a principled
solution via **dynamic weighting**:

```
FL(p_t) = -(1 - p_t)^γ · log(p_t)
```

Where `p_t = exp(-CE)` is the model's probability on the correct class.  
- **Easy examples** (high p_t, e.g. most Neutral tweets): the modulating factor `(1 - p_t)^γ → 0`,
  so their gradient contribution is suppressed.  
- **Hard examples** (low p_t, e.g. minority Positive tweets): factor stays near 1, so they drive
  gradient normally.

This is critical because Positive is only ~11.3 % of the test set. Unlike static class weights,
Focal Loss scales dynamically with how well each sample is already classified.

| γ | Effect |
|---|--------|
| 0 | Identical to standard cross-entropy |
| 1 | Moderate focusing — mild suppression of easy examples |
| 2 | Standard Focal Loss (Lin et al. 2017) — strong focusing |

## Experimental design

- **Model:** XLM-R only (the clean transformer without domain adaptation)
- **Gamma values:** γ=1.0 and γ=2.0 (two separate training runs)
- **Setup:** identical to v2 — clean gold dev set for checkpoint selection, 7 epochs, differential LR
- **Differential LR:** classifier head = 2e-5; backbone = 5e-6 (0.25×)


In [ ]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# os.environ["HF_TOKEN"] = ""  # REMOVED: set HF_TOKEN env var before running

import sys
import sys
from pathlib import Path
_cwd = Path.cwd()
for _p in [_cwd / '../src', _cwd / '../3_Source', _cwd / 'src', _cwd / '3_Source']:
    if (_p / 'config.py').exists():
        sys.path.insert(0, str(_p.resolve()))
        break
else:
    raise ImportError('config.py not found -- run Jupyter from the project root or notebooks/ directory')

import json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from config import (
    SEED, LABEL2ID, ID2LABEL, NUM_LABELS,
    RESULTS_DIR, MODELS_V2, FINETUNE_CONFIG_V2,
    CLEAN_DEV_CSV, CLEAN_TEST_CSV
)

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 4
GRAD_ACCUM = 4
print(f'Device: {DEVICE}, effective batch: {BATCH_SIZE * GRAD_ACCUM}')


In [ ]:
import re

def preprocess(text: str) -> str:
    """Lowercase, remove URLs, normalise whitespace."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load splits — train from stratified split saved by Notebook 01,
# dev and test from the clean gold-label splits (v2 setup)
df_tr   = pd.read_csv(RESULTS_DIR / 'train_split.csv')
df_dev  = pd.read_csv(CLEAN_DEV_CSV)
df_test = pd.read_csv(CLEAN_TEST_CSV)

for df in [df_tr, df_dev, df_test]:
    df['text_clean'] = df['text'].apply(preprocess)
    df['label_id']   = df['label'].map(LABEL2ID)

print(f'Train: {len(df_tr):,}  |  Dev (clean): {len(df_dev):,}  |  Test (clean): {len(df_test):,}')
print(f'Label mapping: {LABEL2ID}')
print(f'Train label dist: {df_tr["label"].value_counts().to_dict()}')
print(f'Test  label dist: {df_test["label"].value_counts().to_dict()}')


In [ ]:
from torch.utils.data import DataLoader

def make_hf_dataset(df, tokenizer, max_length):
    """Convert a DataFrame to a HuggingFace Dataset with tokenised inputs."""
    ds = Dataset.from_dict({
        'text':  df['text_clean'].tolist(),
        'label': df['label_id'].tolist(),
    })
    def tokenize(batch):
        return tokenizer(
            batch['text'],
            truncation=True,
            padding='max_length',
            max_length=max_length,
        )
    ds = ds.map(tokenize, batched=True, batch_size=256)
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    return ds


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {'macro_f1': macro_f1}


def load_json_safe(path):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return None


def evaluate_on_test(model, tokenizer, df_test, max_length, model_key, results_dir):
    """Run inference on the clean test set and save results."""
    model.eval()
    model.to(DEVICE)

    ds_test   = make_hf_dataset(df_test, tokenizer, max_length)
    all_preds = []

    loader = DataLoader(ds_test, batch_size=64)
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)

    label_order = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']
    true_labels = df_test['label'].tolist()
    pred_labels = [ID2LABEL[p] for p in all_preds]

    macro_f1    = f1_score(true_labels, pred_labels, average='macro', labels=label_order)
    report_dict = classification_report(
        true_labels, pred_labels,
        labels=label_order, target_names=label_order,
        output_dict=True
    )

    print(f'\n=== {MODELS_V2[model_key]["name"]} — Clean Test Set ===')
    print(f'Macro-F1: {macro_f1:.4f}')
    print(classification_report(
        true_labels, pred_labels,
        labels=label_order, target_names=label_order
    ))

    # Confusion matrix
    cm  = confusion_matrix(true_labels, pred_labels, labels=label_order)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_order)
    disp.plot(ax=ax, colorbar=True, cmap='Blues')
    ax.set_title(
        f'{MODELS_V2[model_key]["name"]} — Clean Test Set\n'
        f'(Macro-F1 = {macro_f1:.4f}, n={len(df_test):,})',
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig(results_dir / f'{model_key}_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Save JSON results
    results = {
        'model':      MODELS_V2[model_key]['name'],
        'checkpoint': MODELS_V2[model_key]['checkpoint'],
        'macro_f1':   round(macro_f1, 4),
        'per_class':  {
            cls: {
                'precision': round(report_dict[cls]['precision'], 4),
                'recall':    round(report_dict[cls]['recall'],    4),
                'f1':        round(report_dict[cls]['f1-score'],  4),
            }
            for cls in label_order
        }
    }
    out_path = results_dir / f'{model_key}_results.json'
    with open(out_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'Results saved to {out_path}')
    return results


print('Helpers defined.')


In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss for multi-class classification.

    Addresses class imbalance by down-weighting easy (well-classified) examples
    and focusing gradient on hard misclassified examples. Critical for the
    MESocSentiment corpus where Positive class is only 11.3% of test set.

    Reference: Lin et al. (2017) "Focal Loss for Dense Object Detection"
    gamma=0: equivalent to standard cross-entropy
    gamma=1: moderate focusing
    gamma=2: standard setting from original paper, strong focusing on hard examples
    """
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        # logits: (batch_size, num_classes)
        # labels: (batch_size,) — class indices, not one-hot
        ce_loss   = F.cross_entropy(logits, labels, reduction='none')   # (batch,)
        pt        = torch.exp(-ce_loss)                                  # p_t: probability of true class
        focal_loss = (1.0 - pt) ** self.gamma * ce_loss                  # focal modulation
        return focal_loss.mean()


class FocalTrainer(Trainer):
    """
    HuggingFace Trainer with Focal Loss replacing standard cross-entropy.
    """
    def __init__(self, focal_gamma: float = 2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss_fn = FocalLoss(gamma=focal_gamma)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss    = self.focal_loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


print(f'FocalLoss and FocalTrainer defined.')
print(f'  gamma=0 \u2192 standard cross-entropy')
print(f'  gamma=1 \u2192 moderate focusing on hard examples')
print(f'  gamma=2 \u2192 strong focusing (standard from Lin et al. 2017)')


In [ ]:
from torch.optim import AdamW


class FocalDifferentialLRTrainer(FocalTrainer):
    """FocalTrainer + differential learning rates (head=2e-5, backbone=5e-6)."""

    def create_optimizer(self):
        no_decay      = {"bias", "LayerNorm.weight", "layer_norm.weight", "norm.weight"}
        head_keywords = {"classifier", "pooler"}

        head_decay, head_no_decay       = [], []
        backbone_decay, backbone_no_decay = [], []

        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            is_head     = any(kw in name for kw in head_keywords)
            is_no_decay = any(nd in name for nd in no_decay)

            if is_head:
                (head_no_decay if is_no_decay else head_decay).append(param)
            else:
                (backbone_no_decay if is_no_decay else backbone_decay).append(param)

        backbone_lr  = self.args.learning_rate * 0.25
        param_groups = [
            {"params": head_decay,        "lr": self.args.learning_rate, "weight_decay": self.args.weight_decay},
            {"params": head_no_decay,     "lr": self.args.learning_rate, "weight_decay": 0.0},
            {"params": backbone_decay,    "lr": backbone_lr,             "weight_decay": self.args.weight_decay},
            {"params": backbone_no_decay, "lr": backbone_lr,             "weight_decay": 0.0},
        ]
        param_groups = [g for g in param_groups if len(g["params"]) > 0]
        self.optimizer = AdamW(param_groups, eps=1e-8)
        return self.optimizer


print('FocalDifferentialLRTrainer defined.')


In [ ]:
def train_xlm_r_focal(gamma: float) -> dict:
    model_key   = f'xlm_r_focal_g{int(gamma)}'
    cfg         = MODELS_V2[model_key]
    results_dir = cfg['results_dir']
    results_dir.mkdir(parents=True, exist_ok=True)
    output_dir  = results_dir / 'checkpoints'

    print(f'\n{"="*65}')
    print(f'XLM-R with Focal Loss (gamma={gamma})')
    print(f'Results dir: {results_dir}')
    print(f'{"="*65}')

    tokenizer  = AutoTokenizer.from_pretrained(cfg['checkpoint'])
    max_length = FINETUNE_CONFIG_V2['max_seq_length']

    ds_train = make_hf_dataset(df_tr,  tokenizer, max_length)
    ds_dev   = make_hf_dataset(df_dev, tokenizer, max_length)

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg['checkpoint'],
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=FINETUNE_CONFIG_V2['num_train_epochs'],
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=FINETUNE_CONFIG_V2['per_device_eval_batch_size'],
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=FINETUNE_CONFIG_V2['learning_rate'],
        warmup_ratio=FINETUNE_CONFIG_V2['warmup_ratio'],
        weight_decay=FINETUNE_CONFIG_V2['weight_decay'],
        max_grad_norm=FINETUNE_CONFIG_V2['max_grad_norm'],
        eval_strategy=FINETUNE_CONFIG_V2['eval_strategy'],
        save_strategy=FINETUNE_CONFIG_V2['save_strategy'],
        load_best_model_at_end=FINETUNE_CONFIG_V2['load_best_model_at_end'],
        metric_for_best_model=FINETUNE_CONFIG_V2['metric_for_best_model'],
        greater_is_better=True,
        logging_steps=FINETUNE_CONFIG_V2['logging_steps'],
        fp16=(DEVICE == 'cuda'),
        bf16=(DEVICE == 'mps'),
        dataloader_num_workers=FINETUNE_CONFIG_V2['dataloader_num_workers'],
        seed=SEED,
        report_to='none',
        save_total_limit=2,
    )

    trainer = FocalDifferentialLRTrainer(
        focal_gamma=gamma,
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_dev,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print(f'Best clean-dev Macro-F1: {trainer.state.best_metric:.4f}')

    history = trainer.state.log_history
    with open(results_dir / 'training_history.json', 'w') as f:
        json.dump(history, f, indent=2)

    results = evaluate_on_test(trainer.model, tokenizer, df_test, max_length, model_key, results_dir)

    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


print('train_xlm_r_focal() defined.')


In [ ]:
focal_g1_results = train_xlm_r_focal(gamma=1.0)


In [ ]:
focal_g2_results = train_xlm_r_focal(gamma=2.0)


In [ ]:
# Load XLM-R v2 (cross-entropy baseline for fair comparison)
xlm_r_v2 = load_json_safe(RESULTS_DIR / 'xlm_r_v2' / 'xlm_r_results.json')

print('\n=== FOCAL LOSS ABLATION \u2014 XLM-R (all with clean dev, 7 epochs, diff LR) ===')
print(f'{"Config":<30} {"Macro-F1":>10} {"NEG-F1":>8} {"NEU-F1":>8} {"POS-F1":>8}')
print('-' * 70)
for name, r in [
    ('XLM-R v2 (cross-entropy)', xlm_r_v2),
    ('XLM-R + Focal Loss (gamma=1.0)', focal_g1_results),
    ('XLM-R + Focal Loss (gamma=2.0)', focal_g2_results),
]:
    if r:
        print(f'{name:<30} {r["macro_f1"]:>10.4f} '
              f'{r["per_class"]["NEGATIVE"]["f1"]:>8.4f} '
              f'{r["per_class"]["NEUTRAL"]["f1"]:>8.4f} '
              f'{r["per_class"]["POSITIVE"]["f1"]:>8.4f}')
    else:
        print(f'{name:<30} {"(not found)":>10}')

# Save focal results for Notebook 03
all_focal = {
    'xlm_r_focal_g1': focal_g1_results,
    'xlm_r_focal_g2': focal_g2_results,
}
with open(RESULTS_DIR / 'focal_loss_results.json', 'w') as f:
    json.dump(all_focal, f, indent=2)
print('\nResults saved to results/focal_loss_results.json')
